# Data Cleaning & Preprocessing 

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import requests

# Set project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "Data"

### Load Data

In [2]:
# Path to migraine dataset
MIGRAINE_FILE = DATA_DIR / "Raw" / "korean_migraine_study_translated.xlsx"

# Load patient data
patients = pd.read_excel(
    MIGRAINE_FILE,
    sheet_name="patients62"
)

# Load headache diary
headache = pd.read_excel(
    MIGRAINE_FILE,
    sheet_name="headache diary 1099"
)

# Load total diary
diary = pd.read_excel(
    MIGRAINE_FILE,
    sheet_name="total diary 4579"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Dataset found: {MIGRAINE_FILE.exists()}")

Project root: c:\Users\rebec\Documents\Thesis
Data directory: c:\Users\rebec\Documents\Thesis\Data
Dataset found: True


In [3]:
# Create a copy so the raw data remain unchanged
patients_df = patients.copy()

# Verify dimensions
print(f"Rows: {patients_df.shape[0]}")
print(f"Columns: {patients_df.shape[1]}")

# Verify data types before cleaning
patients_df.info()

Rows: 132
Columns: 185
<class 'pandas.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Columns: 185 entries, Unique ID to Total Moderate Exercise Time.1
dtypes: float64(84), object(63), str(38)
memory usage: 190.9+ KB


In [4]:
# Create a copy so the raw data remain unchanged
headache_df = headache.copy()

# Verify dimensions
print(f"Rows: {headache_df.shape[0]}")
print(f"Columns: {headache_df.shape[1]}")

# Verify data types before cleaning
headache_df.info()

Rows: 1100
Columns: 101
<class 'pandas.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Columns: 101 entries, No. to Headache Ongoing (Y/N)
dtypes: float64(58), int64(25), object(1), str(17)
memory usage: 868.1+ KB


In [5]:
# Create a copy so the raw data remain unchanged
diary_df = diary.copy()

# Verify dimensions
print(f"Rows: {diary_df.shape[0]}")
print(f"Columns: {diary_df.shape[1]}")

# Verify data types before cleaning
diary_df.info()

Rows: 4579
Columns: 103
<class 'pandas.DataFrame'>
RangeIndex: 4579 entries, 0 to 4578
Columns: 103 entries, No. to Headache Ongoing (Y/N)
dtypes: float64(73), int64(15), object(1), str(14)
memory usage: 3.6+ MB


In [6]:
duplicate_rows_patients = patients_df.duplicated().sum()
duplicate_rows_headache = headache_df.duplicated().sum()
duplicate_rows_diary = diary_df.duplicated().sum()

print(f"Duplicate rows in patients_df: {duplicate_rows_patients}")
print(f"Duplicate rows in headache_df: {duplicate_rows_headache}")
print(f"Duplicate rows in diary_df: {duplicate_rows_diary}")

Duplicate rows in patients_df: 46
Duplicate rows in headache_df: 0
Duplicate rows in diary_df: 0


In [7]:
patients_df[patients_df.duplicated(keep=False)]

,Unique ID,Registration No.,Study No.,Sex,Age,Hospital,Height,Height Unknown,Weight,Weight Unknown,...,Total Vigorous Exercise Days,Total Moderate Exercise Days,Total Vigorous Exercise Time,Total Moderate Exercise Time,Unnamed: 179,Total Exercise Days.1,Total Vigorous Exercise Days.1,Total Moderate Exercise Days.1,Total Vigorous Exercise Time.1,Total Moderate Exercise Time.1
68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
patients_df = patients_df.drop_duplicates().reset_index(drop=True)

print(f"Duplicate rows after removal: {patients_df.duplicated().sum()}")

Duplicate rows after removal: 0


In [9]:
missing_summary_patients = pd.DataFrame({
    "Missing": patients_df.isna().sum(),
    "Percent Missing": (
        patients_df.isna().mean()*100
    ).round(2)
})

missing_summary_patients = (
    missing_summary_patients
    .sort_values("Percent Missing",
                 ascending=False)
)

missing_summary_patients

,Missing,Percent Missing
Migraine Diagnosis (Other),86,100.00
Medication Type 2,86,100.00
Preventive Treatment Type (Other) 2,86,100.00
Unnamed: 179,86,100.00
Unnamed: 173,86,100.00
...,...,...
Menstrual Cycle: Menstruation.1,13,15.12
Menstrual Cycle: Ovulation.1,13,15.12
Excessive Emotional Change.1,13,15.12
Weather/Temperature Change.1,13,15.12


In [10]:
missing_summary_headache = pd.DataFrame({
    "Missing": headache_df.isna().sum(),
    "Percent Missing": (
        headache_df.isna().mean()*100
    ).round(2)
})

missing_summary_headache = (
    missing_summary_headache
    .sort_values("Percent Missing",
                 ascending=False)
)

missing_summary_headache

,Missing,Percent Missing
Unnamed: 6,1099,99.91
Combination Drug,995,90.45
Other Trigger Factors,951,86.45
Relief Factor (Other),938,85.27
Tylenol (Acetaminophen),931,84.64
...,...,...
External Factor: Inappropriate Lighting,0,0.00
External Factor: Noise,0,0.00
External Factor: Excessive Sunlight,0,0.00
External Factor: Weather/Temperature Change,0,0.00


In [11]:
missing_summary_diary = pd.DataFrame({
    "Missing": diary_df.isna().sum(),
    "Percent Missing": (
        diary_df.isna().mean()*100
    ).round(2)
})

missing_summary_diary = (
    missing_summary_diary
    .sort_values("Percent Missing",
                 ascending=False)
)

missing_summary_diary

,Missing,Percent Missing
Combination Drug,4474,97.71
Relief Factor (Other),4417,96.46
Unnamed: 87,4415,96.42
Tylenol (Acetaminophen),4410,96.31
Triptan,4368,95.39
...,...,...
Internal Factor: Totals,0,0.00
Other Factor: Total,0,0.00
External Factor: Total,0,0.00
Relief Factor: Total,0,0.00


### Create Variable Selection Table

In [12]:
variable_selection = pd.DataFrame({
    "Variable": diary_df.columns,
    "Keep": "",
    "Model": "",
    "Reason": ""
})

variable_selection

variable_selection.to_excel(
    "variable_selection.xlsx",
    index=False
)

### Identifying Number of Headaches That Are Classified as Migraines

In [13]:
headache_df["Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis"].value_counts(dropna=False)

Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis
0.0    763
1.0    336
NaN      1
Name: count, dtype: int64

In [14]:
headache_df.groupby(
    "Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis"
)["Severity"].value_counts()

Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis  Severity
0.0                                                     Mild        468
                                                        Moderate    232
                                                        Severe       35
1.0                                                     Moderate    170
                                                        Severe       82
                                                        Mild         80
Name: count, dtype: int64

### Drop Variables with No Predictive Value

In [15]:
diary_raw = diary.copy()

diary_clean = diary_raw.copy()

# Remove variables you've already decided against
columns_to_remove = [
    "Unique ID",
    "Start Time.1",
    "End Time",
    "End Time.1",
    "Duration",
    "Duration (min)",
    "Date",
    "Severity",
    "Intensity (Mild vs Moderate/Severe)",
    "Pain Left Side",
    "Pain Right Side",
    "Pain Center",
    "Pain Both Sides",
    "Pain Around Eyes",
    "Pain Around Back of Neck",
    "Total Pain Locations",
    "Pain is Unilateral",
    "Throbbing Headache",
    "Tightening Headache",
    "Pounding Headache",
    "Dull/Heavy Headache",
    "Total Headache Types",
    "Is Pulsating",
    "Worsened by Movement",
    "Worsened by Movement.1",
    "Queasiness Accompanying Symptom",
    "Sensitive to Sound Accompanying Symptom",
    "Vomiting Accompanying Symptom",
    "Nausea/Vomiting Accompanying Symptom",
    "Sensitive to Smell Accompanying Symptom",
    "Sensitive to Light Accompanying Symptom",
    "Total Accompanying Symptoms",
    "Light/Sound Sensitivity",
    "Major Premise Present for ICHD-3 Diagnosis",
    "Minor Premise Present for ICHD-3 Diagnosis",
    "Acute Medication Use",
    "Tylenol (Acetaminophen)",
    "Combination Drug",
    "Triptan",
    "Other",
    "Other Medication Name",
    "No Effect from Acute Medication",
    "Headache Reduced by Half or More with Acute Medication",
    "Headache Resolved with Acute Medication",
    "Total Acute Medication Effect",
    "Impacts Daily Life",
    "Impacts Daily Life.1",
    "Missed Work/School",
    "Productivity Reduced by Half or More",
    "Total Impact on Work or School",
    "Unable to Do Housework at All",
    "Housework Efficiency Reduced by Half or More",
    "Total Impact at Home",
    "Social/Leisuire: Planned but Unable to Participate",
    "Trigger Factor",
    "Number of Triggers",
    "Internal Factor: Totals",
    "External Factor: Weather/Temperature Change",
    "External Factor: Total",
    "Other Factor: Total",
    "Unnamed: 87",
    "Headache Ongoing (Y/N)",
    "Relief Factor: Sleep",
    "Relief Factor: Rest",
    "Relief Factor: Massage/Stretching",
    "Relief Factor: Exercise",
    "Relief Factor: Other",
    "Relief Factor: Total",
    "Relief Factor (Other)",
    "Vigorous Exercise (min)",
    "Moderate Exercise (min)",
    "Exercise (Y/N)"
]

# Drop only columns that exist; Excel has a typo in some source files ("Ride" vs "Right")
columns_to_remove = [
    "Unique ID",
    "Start Time.1",
    "End Time",
    "End Time.1",
    "Duration",
    "Duration (min)",
    "Date",
    "Severity",
    "Intensity (Mild vs Moderate/Severe)",
    "Pain Left Side",
    "Pain Right Side",
    "Pain Ride Side",  # source data typo in some sheets
    "Pain Center",
    "Pain Both Sides",
    "Pain Around Eyes",
    "Pain Around Back of Neck",
    "Total Pain Locations",
    "Pain is Unilateral",
    "Throbbing Headache",
    "Tightening Headache",
    "Pounding Headache",
    "Dull/Heavy Headache",
    "Total Headache Types",
    "Is Pulsating",
    "Worsened by Movement",
    "Worsened by Movement.1",
    "Queasiness Accompanying Symptom",
    "Sensitive to Sound Accompanying Symptom",
    "Vomiting Accompanying Symptom",
    "Nausea/Vomiting Accompanying Symptom",
    "Sensitive to Smell Accompanying Symptom",
    "Sensitive to Light Accompanying Symptom",
    "Total Accompanying Symptoms",
    "Light/Sound Sensitivity",
    "Major Premise Present for ICHD-3 Diagnosis",
    "Minor Premise Present for ICHD-3 Diagnosis",
    "Acute Medication Use",
    "Tylenol (Acetaminophen)",
    "Combination Drug",
    "Triptan",
    "Other",
    "Other Medication Name",
    "No Effect from Acute Medication",
    "Headache Reduced by Half or More with Acute Medication",
    "Headache Resolved with Acute Medication",
    "Total Acute Medication Effect",
    "Impacts Daily Life",
    "Impacts Daily Life.1",
    "Missed Work/School",
    "Productivity Reduced by Half or More",
    "Total Impact on Work or School",
    "Unable to Do Housework at All",
    "Housework Efficiency Reduced by Half or More",
    "Total Impact at Home",
    "Social/Leisuire: Planned but Unable to Participate",
    "Trigger Factor",
    "Number of Triggers",
    "Internal Factor: Totals",
    "External Factor: Weather/Temperature Change",
    "External Factor: Total",
    "Other Factor: Total",
    "Unnamed: 87",
    "Headache Ongoing (Y/N)",
    "Relief Factor: Sleep",
    "Relief Factor: Rest",
    "Relief Factor: Massage/Stretching",
    "Relief Factor: Exercise",
    "Relief Factor: Other",
    "Relief Factor: Total",
    "Relief Factor (Other)",
    "Vigorous Exercise (min)",
    "Moderate Exercise (min)",
    "Exercise (Y/N)"
]

diary_clean = diary_clean.drop(
    columns=[col for col in columns_to_remove if col in diary_clean.columns],
    errors="ignore"
)

In [16]:
diary_clean.info()
diary_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 4579 entries, 0 to 4578
Data columns (total 31 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   No.                                                         4579 non-null   int64  
 1   Registration No.                                            4579 non-null   str    
 2   Preventive Medication Use                                   4579 non-null   int64  
 3   Start Time                                                  1098 non-null   float64
 4   Index (VAS Style Score)                                     993 non-null    float64
 5   Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis      1098 non-null   float64
 6   Internal Factor: Stress                                     4578 non-null   float64
 7   Internal Factor: Oversleeping                               4578 non-null   float64
 8   Internal 

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Other Factor: Overeating,Other Factor: Excessive Caffeine,Other Factor: Excessive Smoking,Other Factor: Cheese/Chocolate,Other Factor: Travel,Other Factor: Other,Vigorous Exercise,Moderate Exercise,Total Exercise,Headache-free Day
0,911,cmc-0003,0,2.014103e+11,6.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,N
1,511,dha-0014,0,2.014100e+11,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,N
2,231,cmc-0022,1,2.014092e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N
3,280,cmc-0022,1,2.014092e+11,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N
4,307,cmc-0022,1,2.014092e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N


### Identifying and Removing/Imputing Missing Values

In [17]:
diary_clean[
    diary_clean["Headache-free Day"].isna()
]

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Other Factor: Overeating,Other Factor: Excessive Caffeine,Other Factor: Excessive Smoking,Other Factor: Cheese/Chocolate,Other Factor: Travel,Other Factor: Other,Vigorous Exercise,Moderate Exercise,Total Exercise,Headache-free Day
9,312,cmc-0028,0,2.014092e+11,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,NaN
12,614,cmc-0033,1,2.014101e+11,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,NaN
23,375,dha-0010,1,2.014092e+11,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,NaN
37,994,dha-0010,1,2.014110e+11,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,NaN
80,219,cmc-0022,1,2.014092e+11,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4451,518,dha-0010,1,2.014101e+11,NaN,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,2,NaN
4452,253,dha-0018,1,2.014091e+11,4.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,2,NaN
4540,218,cmc-0025,1,2.014092e+11,7.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2,NaN
4545,4276,cmc-0078,1,2.015031e+11,NaN,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,2,NaN


In [18]:
diary_clean.drop(index=342, inplace=True)

In [19]:
# Headache-free day but headache information exists
diary_clean[
    (diary_clean["Headache-free Day"] == "Yes") &
    (diary_clean["Index (VAS Style Score)"].notna())
]

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Other Factor: Overeating,Other Factor: Excessive Caffeine,Other Factor: Excessive Smoking,Other Factor: Cheese/Chocolate,Other Factor: Travel,Other Factor: Other,Vigorous Exercise,Moderate Exercise,Total Exercise,Headache-free Day


In [20]:
rows_to_remove = diary_clean[
    (diary_clean["Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis"] == 1) &
    (diary_clean["Index (VAS Style Score)"].isna())
]

print(f"Rows to remove: {len(rows_to_remove)}")
rows_to_remove.head()

Rows to remove: 34


,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Other Factor: Overeating,Other Factor: Excessive Caffeine,Other Factor: Excessive Smoking,Other Factor: Cheese/Chocolate,Other Factor: Travel,Other Factor: Other,Vigorous Exercise,Moderate Exercise,Total Exercise,Headache-free Day
269,2775,dha-0057,0,2.015012e+11,NaN,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2,N
277,501,dha-0023,0,2.014100e+11,NaN,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N
286,96,cmc-0011,1,2.014091e+11,NaN,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2,N
290,1137,cmc-0011,1,2.014103e+11,NaN,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2,N
306,2108,cmc-0066,1,2.014122e+11,NaN,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2,N


In [21]:
diary_clean = diary_clean[
    ~(
        (diary_clean["Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis"] == 1) &
        (diary_clean["Index (VAS Style Score)"].isna())
    )
].reset_index(drop=True)

### Assigning Severity Based on VAS Score

| VAS  | Category |
| ---- | -------- |
| 0    | No pain  |
| 1–3  | Mild     |
| 4–6  | Moderate |
| 7–10 | Severe   |


In [22]:
def assign_severity(vas):
    if pd.isna(vas):
        return np.nan
    elif vas <= 3:
        return "Mild"
    elif vas <= 6:
        return "Moderate"
    else:
        return "Severe"

diary_clean["Severity_VAS"] = diary_clean["Index (VAS Style Score)"].apply(assign_severity)

diary_clean["Severity_VAS"].value_counts(dropna=False)

diary_clean.head()

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Other Factor: Excessive Caffeine,Other Factor: Excessive Smoking,Other Factor: Cheese/Chocolate,Other Factor: Travel,Other Factor: Other,Vigorous Exercise,Moderate Exercise,Total Exercise,Headache-free Day,Severity_VAS
0,911,cmc-0003,0,2.014103e+11,6.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,N,Moderate
1,511,dha-0014,0,2.014100e+11,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,N,Severe
2,231,cmc-0022,1,2.014092e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate
3,280,cmc-0022,1,2.014092e+11,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate
4,307,cmc-0022,1,2.014092e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate


In [23]:
diary_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4544 entries, 0 to 4543
Data columns (total 32 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   No.                                                         4544 non-null   int64  
 1   Registration No.                                            4544 non-null   str    
 2   Preventive Medication Use                                   4544 non-null   int64  
 3   Start Time                                                  1064 non-null   float64
 4   Index (VAS Style Score)                                     993 non-null    float64
 5   Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis      1064 non-null   float64
 6   Internal Factor: Stress                                     4544 non-null   float64
 7   Internal Factor: Oversleeping                               4544 non-null   float64
 8   Internal 

In [24]:
diary_clean.head(25)

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Other Factor: Excessive Caffeine,Other Factor: Excessive Smoking,Other Factor: Cheese/Chocolate,Other Factor: Travel,Other Factor: Other,Vigorous Exercise,Moderate Exercise,Total Exercise,Headache-free Day,Severity_VAS
0,911,cmc-0003,0,2.014103e+11,6.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,N,Moderate
1,511,dha-0014,0,2.014100e+11,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,N,Severe
2,231,cmc-0022,1,2.014092e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate
3,280,cmc-0022,1,2.014092e+11,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate
4,307,cmc-0022,1,2.014092e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate
5,1048,cmc-0025,1,2.014103e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate
6,772,cmc-0027,1,2.014102e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate
7,1372,cmc-0027,1,2.014111e+11,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Mild
8,1506,cmc-0027,1,2.014112e+11,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,N,Moderate
9,312,cmc-0028,0,2.014092e+11,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,NaN,Mild


In [25]:
missing_hfd = diary_clean["Headache-free Day"].isna()

diary_clean.loc[missing_hfd, "Headache-free Day"] = np.where(
    diary_clean.loc[missing_hfd, "Severity_VAS"].isin(
        ["Mild", "Moderate", "Severe"]
    ),
    "N",
    "Y"
)

diary_clean["Headache-free Day"].value_counts(dropna=False)

Headache-free Day
Y    3500
N    1044
Name: count, dtype: int64

### Reformatting Date and Time

Headache_Date → merge with weather API

Start_Hour → predictor

Time_of_Day → predictor

Start Time → original datetime reformatted (for traceability)

In [26]:
diary_clean["Start Time"] = pd.to_datetime(
    diary_clean["Start Time"].astype("Int64").astype(str),
    format="%Y%m%d%H%M",
    errors="coerce"
)

diary_clean["Start_Hour"] = diary_clean["Start Time"].dt.hour

diary_clean["Weekday"] = diary_clean["Start Time"].dt.day_name()

diary_clean["Month"] = diary_clean["Start Time"].dt.month

def time_of_day(hour):
    if pd.isna(hour):
        return np.nan
    elif 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

diary_clean["Time_of_Day"] = (
    diary_clean["Start_Hour"].apply(time_of_day)
)

diary_clean["Headache_Date"] = diary_clean["Start Time"].dt.normalize()

In [27]:
diary_clean.head()

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Vigorous Exercise,Moderate Exercise,Total Exercise,Headache-free Day,Severity_VAS,Start_Hour,Weekday,Month,Time_of_Day,Headache_Date
0,911,cmc-0003,0,2014-10-25 18:30:00,6.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,N,Moderate,18.0,Saturday,10.0,Evening,2014-10-25
1,511,dha-0014,0,2014-10-03 14:30:00,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,N,Severe,14.0,Friday,10.0,Afternoon,2014-10-03
2,231,cmc-0022,1,2014-09-20 14:04:00,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1,N,Moderate,14.0,Saturday,9.0,Afternoon,2014-09-20
3,280,cmc-0022,1,2014-09-22 08:57:00,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1,N,Moderate,8.0,Monday,9.0,Morning,2014-09-22
4,307,cmc-0022,1,2014-09-23 18:00:00,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1,N,Moderate,18.0,Tuesday,9.0,Evening,2014-09-23


### Missing VAS Score and Headache/Migraine Events

Found that headache events that had missing VAS scores were not classified as migraines according to ICDH-3. Leaving missing data for now because I will most likely exclude this data later

In [28]:
categorical_cols = [
    "Headache-free Day",
    "Severity_VAS"
]

for col in categorical_cols:
    print(f"\n{col}")
    print(diary_clean[col].value_counts(dropna=False))


Headache-free Day
Headache-free Day
Y    3500
N    1044
Name: count, dtype: int64

Severity_VAS
Severity_VAS
NaN         3551
Moderate     415
Mild         393
Severe       185
Name: count, dtype: int64


In [29]:
headache_no_vas = diary_clean[
    (diary_clean["Headache-free Day"] == "N") &
    (diary_clean["Severity_VAS"].isna())
]

print(len(headache_no_vas))

headache_no_vas

62


,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Vigorous Exercise,Moderate Exercise,Total Exercise,Headache-free Day,Severity_VAS,Start_Hour,Weekday,Month,Time_of_Day,Headache_Date
28,628,dha-0010,1,2014-10-07 14:00:00,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1,N,NaN,14.0,Tuesday,10.0,Afternoon,2014-10-07
34,864,dha-0010,1,2014-10-24 15:00:00,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1,N,NaN,15.0,Friday,10.0,Afternoon,2014-10-24
38,420,dha-0013,0,2014-09-28 11:00:00,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1,N,NaN,11.0,Sunday,9.0,Morning,2014-09-28
39,759,dha-0013,0,2014-10-20 13:00:00,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1,N,NaN,13.0,Monday,10.0,Afternoon,2014-10-20
40,1057,dha-0013,0,2014-11-07 17:50:00,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1,N,NaN,17.0,Friday,11.0,Evening,2014-11-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4117,103,dha-0010,1,2014-08-29 11:51:00,NaN,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,2,N,NaN,11.0,Friday,8.0,Morning,2014-08-29
4118,323,dha-0013,0,2014-09-17 15:49:00,NaN,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2,N,NaN,15.0,Wednesday,9.0,Afternoon,2014-09-17
4128,2684,dha-0044,0,2015-01-21 13:30:00,NaN,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,2,N,NaN,13.0,Wednesday,1.0,Afternoon,2015-01-21
4384,1575,cmc-0008,0,2014-11-19 12:00:00,NaN,0.0,1.0,0.0,0.0,0.0,...,1.0,1.0,2,N,NaN,12.0,Wednesday,11.0,Afternoon,2014-11-19


### Converting Binary Features and Total Exercise to Int8

Total exercise is ordinal 0 < 1 < 2 combining vigorous and moderate exercise

In [30]:
binary_cols = [
    "Preventive Medication Use",
    "Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis",
    "Internal Factor: Stress",
    "Internal Factor: Oversleeping",
    "Internal Factor: Sleep Deprivation",
    "Internal Factor: Exercise",
    "Internal Factor: Lack of Exercise",
    "Internal Factor: Physical Fatigue",
    "Internal Factor: Menstrual Cycle: Menstruation",
    "Internal Factor: Menstrual Cycle: Ovulation",
    "Internal Factor: Excessive Emotional Change",
    "External Factor: Excessive Sunlight",
    "External Factor: Noise",
    "External Factor: Inappropriate Lighting",
    "External Factor: Specific Smell (cosmetics, perfume, etc.)",
    "Other Factor: Excessive Alcohol",
    "Other Factor: Irregular Meals (fasting, etc.)",
    "Other Factor: Overeating",
    "Other Factor: Excessive Caffeine",
    "Other Factor: Excessive Smoking",
    "Other Factor: Cheese/Chocolate",
    "Other Factor: Travel",
    "Other Factor: Other",
    "Vigorous Exercise",
    "Moderate Exercise"
]

for col in binary_cols:
    values = set(diary_clean[col].dropna().unique())

    if not values.issubset({0, 1, 0.0, 1.0}):
        print(f"{col}: unexpected values {values}")

for col in binary_cols:
    diary_clean[col] = diary_clean[col].astype("Int8")

diary_clean["Headache_free_binary"] = (
    diary_clean["Headache-free Day"]
    .map({"N": 0, "Y": 1})
    .astype("Int8")
)

In [31]:
pd.crosstab(
    [diary_clean["Vigorous Exercise"],
     diary_clean["Moderate Exercise"]],
    diary_clean["Total Exercise"]
)

Total Exercise                        0    1     2
Vigorous Exercise Moderate Exercise               
0                 0                  41    0     0
                  1                   0  831     0
1                 0                   0   99     0
                  1                   0    0  3573

In [32]:
diary_clean["Total Exercise"].value_counts().sort_index()

Total Exercise
0      41
1     930
2    3573
Name: count, dtype: int64

In [33]:
diary_clean["Total Exercise"] = (
    diary_clean["Total Exercise"]
    .astype("int8")
)

### Save data to CSV and Excel

In [34]:
diary_clean.head()

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Moderate Exercise,Total Exercise,Headache-free Day,Severity_VAS,Start_Hour,Weekday,Month,Time_of_Day,Headache_Date,Headache_free_binary
0,911,cmc-0003,0,2014-10-25 18:30:00,6.0,0,0,0,0,0,...,0,0,N,Moderate,18.0,Saturday,10.0,Evening,2014-10-25,0
1,511,dha-0014,0,2014-10-03 14:30:00,7.0,0,0,0,0,0,...,0,0,N,Severe,14.0,Friday,10.0,Afternoon,2014-10-03,0
2,231,cmc-0022,1,2014-09-20 14:04:00,4.0,0,0,0,0,0,...,1,1,N,Moderate,14.0,Saturday,9.0,Afternoon,2014-09-20,0
3,280,cmc-0022,1,2014-09-22 08:57:00,5.0,0,0,0,0,0,...,1,1,N,Moderate,8.0,Monday,9.0,Morning,2014-09-22,0
4,307,cmc-0022,1,2014-09-23 18:00:00,4.0,0,0,0,0,0,...,1,1,N,Moderate,18.0,Tuesday,9.0,Evening,2014-09-23,0


In [38]:
# Save CSV
diary_clean.to_csv(
    "migraine_daily_diary_clean.csv",
    index=False
)

# Save Excel
diary_clean.to_excel(
    "migraine_daily_diary_clean.xlsx",
    index=False
)